# Nextgen doğal sohbet + RAG LLM eğitimi (BPE, d=256)

Dosya seçmek YOK — her şey GitHub'dan klonlanır (`corpus.jsonl`, tokenizer, intents, kod dahil).

## Adımlar
1. **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. **Çalışma Zamanı → Tümünü çalıştır.**
3. Oturum kesilirse (Colab ~12 sa): **hepsini yeniden çalıştırın** — eğitim kaldığı yerden
   **Drive'da** saklanan checkpoint'ten devam eder (veri parmak izi uyarsa).
4. Tamamen baştan istiyorsan Drive'daki `NextgenAI_llm/llm_ckpt.pt` dosyasını sil.
5. Çıktılar: `MyDrive/NextgenAI_llm/` içinde `llm_model.json` + `llm_model_weights.npz` —
   son hücre ikisini tek zip'e alıp indirir. Yerelde ikisini `model/` klasörüne kopyala.

In [ ]:
import torch
print('torch :', torch.__version__)
print('GPU   :', 'T4 kullaniliyor' if torch.cuda.is_available() else 'YOK - menuden T4 GPU secin')
if torch.cuda.is_available():
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
import os
!rm -rf /content/NextgenAI            # taze kod (fasiniz engellenmez)
!git clone --quiet https://github.com/dozagoat47-star/NextgenAI.git
%cd /content/NextgenAI
!python -m pip install --quiet numpy
from google.colab import drive
drive.mount('/content/drive')         # kalicilik: checkpoint ve cikti burada durur
!mkdir -p /content/drive/MyDrive/NextgenAI_llm

In [ ]:
%%time
# Deneme: veri hatti dogrulamasi (~1-2 dk, GPU gerekmez)
!python train_llm.py --dry-run --rag --kb-map knowledge_map.jsonl --natural 5

In [ ]:
%%time
%cd /content/NextgenAI
import os
os.environ['SAVE_DIR'] = '/content/drive/MyDrive/NextgenAI_llm'   # ckpt + cikti Drive'da
EPOCHS = 80                                                       # patience otomatik durur
!python train_llm.py --rag --kb-map knowledge_map.jsonl --natural 5 --epochs {EPOCHS} --batch-size 64 --val-every 2
# Kesilip devam: hucr 1-5'i tekrar calistir -> ayni Drive ckpt'inden resume

In [ ]:
import os, zipfile
src = '/content/drive/MyDrive/NextgenAI_llm'
for fn in ('llm_ckpt.pt', 'llm_model.json', 'llm_model_weights.npz'):
    p = os.path.join(src, fn)
    if os.path.exists(p):
        print('Drive:', fn, '->', round(os.path.getsize(p)/1e6, 1), 'MB')
z = '/content/nde-irma.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ('llm_model.json', 'llm_model_weights.npz'):
        p = os.path.join(src, fn)
        if os.path.exists(p):
            zf.write(p, fn)
print('Hazir:', z)
from google.colab import files
files.download(z)